In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds -> UTC)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s', utc=True)

# Ensure numeric values where applicable
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) metric.csv: group by (cmdb_id, kpi_name) and compute count, min, max, mean, P95
metric_grouped_full = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)
metric_top20 = metric_grouped_full.sort_values('count', ascending=False).head(20)
metric_cmdb_ids = pd.Series(metric_df['cmdb_id'].unique(), name='cmdb_id')

# 2) trace.csv: group by (cmdb_id, trace_name) and compute same aggregates
trace_grouped_full = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)
trace_top20 = trace_grouped_full.sort_values('count', ascending=False).head(20)
trace_cmdb_ids = pd.Series(trace_df['cmdb_id'].unique(), name='cmdb_id')

# 3) log.csv: group by (cmdb_id, log_name) and compute same aggregates
log_grouped_full = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)
log_top20 = log_grouped_full.sort_values('count', ascending=False).head(20)
log_cmdb_ids = pd.Series(log_df['cmdb_id'].unique(), name='cmdb_id')

# 4) error_logs.csv: total rows, unique cmdb_ids, per-cmdb counts, earliest and latest timestamp
error_total_rows = len(error_logs_df)
error_cmdb_ids = pd.Series(error_logs_df['cmdb_id'].unique(), name='cmdb_id')
error_counts_full = error_logs_df.groupby('cmdb_id', dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
error_counts_head = error_counts_full.head(20)  # compact display
error_earliest = error_logs_df['timestamp'].min()
error_latest = error_logs_df['timestamp'].max()

# Display compact outputs (aggregate tables limited to 20 rows each and compact summaries)
metric_top20, metric_cmdb_ids, trace_top20, trace_cmdb_ids, log_top20, log_cmdb_ids, error_total_rows, error_cmdb_ids, error_counts_head, error_earliest, error_latest

```
Out[1]:
```
```python
# Reuse existing grouped/top variables from previous run: 
# metric_top20, metric_cmdb_ids, trace_top20, trace_cmdb_ids, log_top20, log_cmdb_ids, error_total_rows

# Build a concise plain-English summary using those variables
top_metric_services = ", ".join(pd.Series(metric_top20['cmdb_id'].unique()).head(6).tolist())
top_log_services = ", ".join(pd.Series(log_top20['cmdb_id'].unique()).head(6).tolist())
top_trace_examples = "; ".join(
    trace_top20.head(5).apply(lambda r: f"{r['cmdb_id']}:{r['trace_name'].split('.')[-1]}", axis=1).tolist()
)

summary = (
    "Summary of telemetry (plain English):\n"
    f"- metric.csv: {metric_cmdb_ids.shape[0]} unique cmdb_id values. "
    f"Top metric groups (by sample count) come from services such as: {top_metric_services}. "
    "All top metric groups show 25 samples each. Notable high-magnitude KPIs include large memory values (e.g., ts-travel-service mem ~2.7e8, ts-travel2-mongo mem ~6.8e7) "
    "and substantial disk I/O for some mongo/travel services.\n"
    f"- trace.csv: {trace_cmdb_ids.shape[0]} unique cmdb_id values. Examples of top trace entries: {top_trace_examples}. "
    "Top traces also have 25 samples; some p95 durations reach ~0.8s for admin-travel traces.\n"
    f"- log.csv: {log_cmdb_ids.shape[0]} unique cmdb_id values. Top log sources include: {top_log_services}. "
    "High log row_count observed for ts-basic-service (mean ~2388) and other services; log.error_count in the top groups is zero.\n"
    f"- error_logs.csv: {int(error_total_rows)} rows (empty). No error-log entries present in this dataset.\n\n"
    "Conclusion: telemetry coverage is broad and consistent (25-sample series for top KPIs/traces). "
    "For any further root-cause work, prioritize services with high memory and disk I/O observed (e.g., ts-travel-service, ts-travel2-mongo, ts-travel2-service, ts-admin-basic-info-service)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           min           max          mean           p95
355           ts-voucher-service      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  3.353509e-01  9.272340e-01  4.989553e-01  7.398317e-01
1    ts-admin-basic-info-service      diskio     25  3.503435e+04  3.263982e+05  1.016367e+05  2.784496e+05
2    ts-admin-basic-info-service  latency-50     25  1.578721e-02  1.808015e-02  1.688034e-02  1.766364e-02
316             ts-travel2-mongo      diskio     25  3.299654e+05  5.223012e+05  4.057651e+05  4.856937e+05
317             ts-travel2-mongo         mem     25  6.804275e+07  6.832551e+07  6.816285e+07  6.828267e+07
318             ts-travel2-mongo      socket     25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00
319           ts-travel2-service         cpu     25  4.308528e+00  5.252607e+00  4.811559e+00  5.238207e+00
320           ts-travel2-service      diskio     25  3.575756e+04  6.496395e+04  5.131289e+04  6.443302e+04
321           ts-travel2-service  latency-50     25  1.567027e-02  2.030294e-02  1.822747e-02  2.026588e-02
322           ts-travel2-service  latency-90     25  3.460312e-01  3.843749e-01  3.646646e-01  3.829931e-01
323           ts-travel2-service         mem     25  2.820134e+08  2.826648e+08  2.823435e+08  2.826524e+08
308            ts-travel-service         cpu     25  1.892405e+00  4.871256e+00  3.070147e+00  4.615870e+00
309            ts-travel-service      diskio     25  3.842526e+04  6.019355e+04  5.187548e+04  5.855691e+04
310            ts-travel-service  latency-50     25  1.316158e-02  1.798637e-02  1.563009e-02  1.746523e-02
311            ts-travel-service  latency-90     25  8.725211e-02  3.007803e-01  1.704333e-01  2.928011e-01
312            ts-travel-service         mem     25  2.707866e+08  2.743773e+08  2.718610e+08  2.730814e+08
313            ts-travel-service      socket     25  1.400000e+01  1.738333e+01  1.534833e+01  1.679333e+01
314            ts-travel-service    workload     25  5.813300e+00  1.163606e+01  8.882781e+00  1.058388e+01
315             ts-travel2-mongo         cpu     25  6.837364e-01  8.920318e-01  7.867500e-01  8.856483e-01, 0      ts-admin-basic-info-service
1           ts-admin-order-service
2           ts-admin-route-service
3          ts-admin-travel-service
4            ts-admin-user-service
                  ...             
63                   ts-user-mongo
64                 ts-user-service
65    ts-verification-code-service
66                ts-voucher-mysql
67              ts-voucher-service
Name: cmdb_id, Length: 68, dtype: object,                 cmdb_id                                         trace_name  count         min         max        mean         p95
775     ts-user-service                 trace.to_ts-user-service.row_count     25    4.000000   22.000000   13.120000   20.000000
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25    0.009920    0.017096    0.011041    0.015288
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25    0.010702    0.048802    0.015407    0.032490
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
3                  root     trace.to_ts-admin-basic-info-service.row_count     25    6.000000   24.000000   18.800000   22.800000
4                  root     trace.to_ts-admin-travel-service.duration_mean     25    0.125213    0.408885    0.218724    0.396039
5                  root      trace.to_ts-admin-travel-service.duration_p95     25    0.132727    0.819557    0.390961    0.794774
6                  root        trace.to_ts-admin-travel-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
736  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25    0.004823    0.009647    0.006174    0.008704
737  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25    0.006246    0.052566    0.012591    0.035318
738  ts-travel2-service         trace.to_ts-order-other-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
739  ts-travel2-service          trace.to_ts-order-other-service.row_count     25   24.000000   86.000000   76.120000   85.800000
740  ts-travel2-service            trace.to_ts-route-service.duration_mean     25    0.002891    0.003267    0.003078    0.003218
741  ts-travel2-service             trace.to_ts-route-service.duration_p95     25    0.003525    0.004017    0.003804    0.003967
742  ts-travel2-service               trace.to_ts-route-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
743  ts-travel2-service                trace.to_ts-route-service.row_count     25  177.000000  640.000000  571.920000  635.800000
728  ts-travel2-service           trace.from_ts-seat-service.duration_mean     25    0.009541    0.017509    0.012642    0.016452
729  ts-travel2-service            trace.from_ts-seat-service.duration_p95     25    0.011401    0.069927    0.031929    0.062004
730  ts-travel2-service              trace.from_ts-seat-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
731  ts-travel2-service               trace.from_ts-seat-service.row_count     25  100.000000  352.000000  309.760000  349.600000, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23              ts-travel-service
24             ts-travel2-service
25                ts-user-service
26       ts-consign-price-service
27             ts-consign-service
Name: cmdb_id, Length: 28, dtype: object,                         cmdb_id         log_name  count  min   max     mean     p95
0   ts-admin-basic-info-service  log.error_count     25    0     0     0.00     0.0
1   ts-admin-basic-info-service    log.row_count     25    9    36    28.20    34.6
2       ts-admin-travel-service  log.error_count     25    0     0     0.00     0.0
3       ts-admin-travel-service    log.row_count     25    8    35    28.16    33.0
7          ts-assurance-service    log.row_count     25    4    15     9.60    15.0
6          ts-assurance-service  log.error_count     25    0     0     0.00     0.0
13             ts-basic-service    log.row_count     25  837  2862  2387.80  2843.6
12             ts-basic-service  log.error_count     25    0     0     0.00     0.0
11              ts-auth-service    log.row_count     25   86   358   320.88   352.0
10              ts-auth-service  log.error_count     25    0     0     0.00     0.0
25          ts-contacts-service    log.row_count     25   12    66    39.36    60.0
24          ts-contacts-service  log.error_count     25    0     0     0.00     0.0
64             ts-route-service  log.error_count     25    0     0     0.00     0.0
65             ts-route-service    log.row_count     25  318  1149   979.88  1129.4
57          ts-preserve-service    log.row_count     25   26   197   107.32   186.8
56          ts-preserve-service  log.error_count     25    0     0     0.00     0.0
47             ts-order-service    log.row_count     25  132   475   357.96   469.8
46             ts-order-service  log.error_count     25    0     0     0.00     0.0
45       ts-order-other-service    log.row_count     25  142   522   435.28   508.4
44       ts-order-other-service  log.error_count     25    0     0     0.00     0.0, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2            ts-assurance-service
3                 ts-auth-service
4                ts-basic-service
                 ...             
40         ts-ticket-office-mongo
41                 ts-train-mongo
42                ts-travel-mongo
43               ts-travel2-mongo
44                  ts-user-mongo
Name: cmdb_id, Length: 45, dtype: object, 0, Series([], Name: cmdb_id, dtype: object), Empty DataFrame
Columns: [cmdb_id, count]
Index: [], NaT, NaT)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise load
try:
    metric_df
    trace_df
    log_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    trace_df = pd.read_csv("trace.csv")
    log_df = pd.read_csv("log.csv")

# Parse timestamps as UTC (Unix seconds)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Services of interest
services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 11:21:47", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 11:51:47", tz="UTC")

# --- 1) Metrics ---

# Subset metrics to the five services
metric_subset = metric_df[metric_df['cmdb_id'].isin(services)].copy()

# Compute global thresholds (full series) per (cmdb_id, kpi_name)
metric_thresholds_full = (
    metric_subset
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(full_count='count',
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Filter metric rows to the incident window
metric_window = metric_subset[(metric_subset['timestamp'] >= window_start) & (metric_subset['timestamp'] <= window_end)].copy()

# Merge window rows with global thresholds so each row knows its group's P95/P05
if not metric_thresholds_full.empty:
    metric_window = metric_window.merge(metric_thresholds_full[['cmdb_id','kpi_name','p95','p05']], on=['cmdb_id','kpi_name'], how='left')
else:
    metric_window['p95'] = np.nan
    metric_window['p05'] = np.nan

# Flags for anomalies in window
if not metric_window.empty:
    metric_window['ge_p95'] = metric_window['value'] >= metric_window['p95']
    metric_window['le_p05'] = metric_window['value'] <= metric_window['p05']
else:
    metric_window['ge_p95'] = pd.Series(dtype=bool)
    metric_window['le_p05'] = pd.Series(dtype=bool)

# Aggregate window-level anomaly counts and earliest timestamps
if not metric_window.empty:
    metric_window_counts = (
        metric_window
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(window_count=('value','count'),
             anomalies_ge_p95_count=('ge_p95','sum'),
             anomalies_le_p05_count=('le_p05','sum'))
        .reset_index()
    )
    # earliest timestamps where conditions hold
    earliest_ge = (metric_window[metric_window['ge_p95']]
                   .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_ge_p95_ts'))
    earliest_le = (metric_window[metric_window['le_p05']]
                   .groupby(['cmdb_id','kpi_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_le_p05_ts'))
    # Merge counts and earliest timestamps
    metric_window_agg = metric_window_counts.merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left').merge(earliest_le, on=['cmdb_id','kpi_name'], how='left')
else:
    metric_window_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count','anomalies_ge_p95_count','anomalies_le_p05_count','earliest_ge_p95_ts','earliest_le_p05_ts'])

# Combine full thresholds and window aggregates
metric_final = metric_thresholds_full.merge(metric_window_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NA counts with 0 and keep earliest timestamps as NaT where missing
for c in ['window_count','anomalies_ge_p95_count','anomalies_le_p05_count']:
    if c in metric_final.columns:
        metric_final[c] = metric_final[c].fillna(0).astype(int)
# Ensure earliest timestamp columns are timezone-aware or NaT
if 'earliest_ge_p95_ts' in metric_final.columns:
    metric_final['earliest_ge_p95_ts'] = pd.to_datetime(metric_final['earliest_ge_p95_ts'], utc=True)
else:
    metric_final['earliest_ge_p95_ts'] = pd.NaT
if 'earliest_le_p05_ts' in metric_final.columns:
    metric_final['earliest_le_p05_ts'] = pd.to_datetime(metric_final['earliest_le_p05_ts'], utc=True)
else:
    metric_final['earliest_le_p05_ts'] = pd.NaT

# Select and order columns as requested
metric_final_table = metric_final[['cmdb_id','kpi_name','full_count','p95','p05','window_count','anomalies_ge_p95_count','earliest_ge_p95_ts','anomalies_le_p05_count','earliest_le_p05_ts']]

# List of unique kpi_name values for these five services
metric_kpi_names = pd.Series(metric_subset['kpi_name'].unique(), name='kpi_name')

# Keep full grouped thresholds in kernel variable: metric_thresholds_full (already set)

# --- 2) Traces ---

trace_subset = trace_df[trace_df['cmdb_id'].isin(services)].copy()

trace_thresholds_full = (
    trace_subset
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(full_count='count',
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

trace_window = trace_subset[(trace_subset['timestamp'] >= window_start) & (trace_subset['timestamp'] <= window_end)].copy()

if not trace_thresholds_full.empty:
    trace_window = trace_window.merge(trace_thresholds_full[['cmdb_id','trace_name','p95','p05']], on=['cmdb_id','trace_name'], how='left')
else:
    trace_window['p95'] = np.nan
    trace_window['p05'] = np.nan

if not trace_window.empty:
    trace_window['ge_p95'] = trace_window['value'] >= trace_window['p95']
    trace_window['le_p05'] = trace_window['value'] <= trace_window['p05']
else:
    trace_window['ge_p95'] = pd.Series(dtype=bool)
    trace_window['le_p05'] = pd.Series(dtype=bool)

if not trace_window.empty:
    trace_window_counts = (
        trace_window
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(window_count=('value','count'),
             anomalies_ge_p95_count=('ge_p95','sum'),
             anomalies_le_p05_count=('le_p05','sum'))
        .reset_index()
    )
    earliest_ge = (trace_window[trace_window['ge_p95']]
                   .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_ge_p95_ts'))
    earliest_le = (trace_window[trace_window['le_p05']]
                   .groupby(['cmdb_id','trace_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_le_p05_ts'))
    trace_window_agg = trace_window_counts.merge(earliest_ge, on=['cmdb_id','trace_name'], how='left').merge(earliest_le, on=['cmdb_id','trace_name'], how='left')
else:
    trace_window_agg = pd.DataFrame(columns=['cmdb_id','trace_name','window_count','anomalies_ge_p95_count','anomalies_le_p05_count','earliest_ge_p95_ts','earliest_le_p05_ts'])

trace_final = trace_thresholds_full.merge(trace_window_agg, on=['cmdb_id','trace_name'], how='left')

for c in ['window_count','anomalies_ge_p95_count','anomalies_le_p05_count']:
    if c in trace_final.columns:
        trace_final[c] = trace_final[c].fillna(0).astype(int)

if 'earliest_ge_p95_ts' in trace_final.columns:
    trace_final['earliest_ge_p95_ts'] = pd.to_datetime(trace_final['earliest_ge_p95_ts'], utc=True)
else:
    trace_final['earliest_ge_p95_ts'] = pd.NaT
if 'earliest_le_p05_ts' in trace_final.columns:
    trace_final['earliest_le_p05_ts'] = pd.to_datetime(trace_final['earliest_le_p05_ts'], utc=True)
else:
    trace_final['earliest_le_p05_ts'] = pd.NaT

trace_final_table = trace_final[['cmdb_id','trace_name','full_count','p95','p05','window_count','anomalies_ge_p95_count','earliest_ge_p95_ts','anomalies_le_p05_count','earliest_le_p05_ts']]

trace_names = pd.Series(trace_subset['trace_name'].unique(), name='trace_name')

# Keep trace_thresholds_full in kernel

# --- 3) Logs ---

log_subset = log_df[log_df['cmdb_id'].isin(services)].copy()

log_thresholds_full = (
    log_subset
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(full_count='count',
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

log_window = log_subset[(log_subset['timestamp'] >= window_start) & (log_subset['timestamp'] <= window_end)].copy()

if not log_thresholds_full.empty:
    log_window = log_window.merge(log_thresholds_full[['cmdb_id','log_name','p95','p05']], on=['cmdb_id','log_name'], how='left')
else:
    log_window['p95'] = np.nan
    log_window['p05'] = np.nan

if not log_window.empty:
    log_window['ge_p95'] = log_window['value'] >= log_window['p95']
    log_window['le_p05'] = log_window['value'] <= log_window['p05']
else:
    log_window['ge_p95'] = pd.Series(dtype=bool)
    log_window['le_p05'] = pd.Series(dtype=bool)

if not log_window.empty:
    log_window_counts = (
        log_window
        .groupby(['cmdb_id','log_name'], dropna=False)
        .agg(window_count=('value','count'),
             anomalies_ge_p95_count=('ge_p95','sum'),
             anomalies_le_p05_count=('le_p05','sum'))
        .reset_index()
    )
    earliest_ge = (log_window[log_window['ge_p95']]
                   .groupby(['cmdb_id','log_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_ge_p95_ts'))
    earliest_le = (log_window[log_window['le_p05']]
                   .groupby(['cmdb_id','log_name'], dropna=False)['timestamp']
                   .min().reset_index(name='earliest_le_p05_ts'))
    log_window_agg = log_window_counts.merge(earliest_ge, on=['cmdb_id','log_name'], how='left').merge(earliest_le, on=['cmdb_id','log_name'], how='left')
else:
    log_window_agg = pd.DataFrame(columns=['cmdb_id','log_name','window_count','anomalies_ge_p95_count','anomalies_le_p05_count','earliest_ge_p95_ts','earliest_le_p05_ts'])

log_final = log_thresholds_full.merge(log_window_agg, on=['cmdb_id','log_name'], how='left')

for c in ['window_count','anomalies_ge_p95_count','anomalies_le_p05_count']:
    if c in log_final.columns:
        log_final[c] = log_final[c].fillna(0).astype(int)

if 'earliest_ge_p95_ts' in log_final.columns:
    log_final['earliest_ge_p95_ts'] = pd.to_datetime(log_final['earliest_ge_p95_ts'], utc=True)
else:
    log_final['earliest_ge_p95_ts'] = pd.NaT
if 'earliest_le_p05_ts' in log_final.columns:
    log_final['earliest_le_p05_ts'] = pd.to_datetime(log_final['earliest_le_p05_ts'], utc=True)
else:
    log_final['earliest_le_p05_ts'] = pd.NaT

log_final_table = log_final[['cmdb_id','log_name','full_count','p95','p05','window_count','anomalies_ge_p95_count','earliest_ge_p95_ts','anomalies_le_p05_count','earliest_le_p05_ts']]

log_names = pd.Series(log_subset['log_name'].unique(), name='log_name')

# Keep full grouped DataFrames and threshold series in kernel variables:
# metric_thresholds_full, trace_thresholds_full, log_thresholds_full

# Compact outputs (limit to at most 50 rows each)
metric_out = metric_final_table.sort_values(['full_count','cmdb_id','kpi_name'], ascending=[False,True,True]).head(50)
trace_out  = trace_final_table.sort_values(['full_count','cmdb_id','trace_name'], ascending=[False,True,True]).head(50)
log_out    = log_final_table.sort_values(['full_count','cmdb_id','log_name'], ascending=[False,True,True]).head(50)

# Display the compact results and the unique name lists
metric_out, metric_kpi_names, trace_out, trace_names, log_out, log_names

```
Out[2]:
```
```python
# Compose a plain-English summary of the analysis results (display as a single string)
summary = (
    "Analysis summary for services: ts-auth-service, ts-order-service, ts-route-service, "
    "ts-train-service, ts-travel-service (global thresholds computed from full series; "
    "incident window = 2024-01-23 11:21:47 to 2024-01-23 11:51:47 UTC):\n\n"
    "1) Metrics:\n"
    "- All (cmdb_id, kpi_name) groups for these services have full_count = 25 samples (full series available).\n"
    "- Within the incident window every group also has 25 samples (window_count = 25).\n"
    "- For most metric KPIs, there are a small number of points that exceed the global P95 (typically 2 points) "
    "and a small number that fall at-or-below the global P05 (typically 2 points). "
    "Earliest over-P95 occurrences are generally between ~11:24 and ~11:47 UTC depending on KPI; earliest under-P05 around ~11:24–11:28 UTC.\n"
    "- Notable high-value KPIs observed: memory for ts-travel-service (p95 ~2.73e8) and ts-auth-service (p95 ~2.40e8). "
    "Disk I/O and CPU spikes also show a couple of outlier points across services.\n\n"
    "2) Traces:\n"
    "- Trace groups (full_count = 25 each) similarly show small numbers of values above P95 and below P05 in-window (commonly 2 each). "
    "Earliest over/under threshold timestamps are typically within the same window (~11:24–11:40 UTC).\n"
    "- Several trace error_rate series are constant-zero; because p95 and p05 are 0, these series appear as anomalies by the >=p95 and <=p05 logic (counts = 25). "
    "This reflects that error_rate is uniformly zero rather than indicating many actual errors.\n"
    "- Some trace p95 durations reach notable values (examples seen earlier up to ~0.8s for certain admin-travel traces).\n\n"
    "3) Logs:\n"
    "- log.error_count is zero across the samples for these services (p95 = p05 = 0), so our >=P95 and <=P05 checks both count 25—this indicates no error-log entries, not an error surge.\n"
    "- log.row_count shows occasional high points (e.g., a few timestamps exceeding p95); earliest over-p95 log row_count timestamps are around ~11:26–11:34 UTC depending on service.\n"
    "- Services with larger log volume in this set include ts-travel-service and ts-route-service (higher row_count means more log traffic, with a couple of peak samples).\n\n"
    "Overall conclusion and recommendation:\n"
    "- There are modest, short-lived spikes in CPU/diskio/memory and trace durations (usually 1–2 datapoints) during the incident window across these five services. "
    "Memory p95 values for ts-travel-service and ts-auth-service stand out as high absolute values.\n"
    "- Error log rates are effectively zero; trace error_rate being zero is expected and should not be treated as a failure signal here.\n"
    "- For root-cause follow-up, inspect the specific timestamps reported as earliest_ge_p95_ts for each metric/trace (they fall in the 11:24–11:47 UTC range) and correlate with application logs or infrastructure events at those times. "
    "Focus initial investigation on memory and CPU/diskio spikes for ts-travel-service and ts-auth-service.\n\n"
    "If you want, I can produce a compact list of the exact earliest over-P95 timestamps per KPI/trace/log (one table each) for direct drill-down."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  full_count           p95           p05  window_count  anomalies_ge_p95_count        earliest_ge_p95_ts  anomalies_le_p05_count        earliest_le_p05_ts
0     ts-auth-service         cpu          25  1.672277e+01  1.316083e+01            25                       2 2024-01-23 11:32:00+00:00                       2 2024-01-23 11:45:00+00:00
1     ts-auth-service      diskio          25  6.105997e+04  4.315461e+04            25                       2 2024-01-23 11:24:00+00:00                       2 2024-01-23 11:28:00+00:00
2     ts-auth-service  latency-50          25  2.769604e-01  2.125969e-01            25                       2 2024-01-23 11:37:00+00:00                       2 2024-01-23 11:27:00+00:00
3     ts-auth-service  latency-90          25  1.000781e+00  4.634286e-01            25                       2 2024-01-23 11:39:00+00:00                       2 2024-01-23 11:24:00+00:00
4     ts-auth-service         mem          25  2.399867e+08  2.369201e+08            25                       2 2024-01-23 11:47:00+00:00                       2 2024-01-23 11:24:00+00:00
..                ...         ...         ...           ...           ...           ...                     ...                       ...                     ...                       ...
31  ts-travel-service  latency-50          25  1.746523e-02  1.379154e-02            25                       2 2024-01-23 11:38:00+00:00                       2 2024-01-23 11:27:00+00:00
32  ts-travel-service  latency-90          25  2.928011e-01  9.127215e-02            25                       2 2024-01-23 11:38:00+00:00                       2 2024-01-23 11:26:00+00:00
33  ts-travel-service         mem          25  2.730814e+08  2.710320e+08            25                       2 2024-01-23 11:42:00+00:00                       2 2024-01-23 11:24:00+00:00
34  ts-travel-service      socket          25  1.679333e+01  1.404333e+01            25                       2 2024-01-23 11:27:00+00:00                       2 2024-01-23 11:24:00+00:00
35  ts-travel-service    workload          25  1.058388e+01  7.068670e+00            25                       2 2024-01-23 11:43:00+00:00                       2 2024-01-23 11:28:00+00:00

[36 rows x 10 columns], 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
7         error
Name: kpi_name, dtype: object,              cmdb_id                                   trace_name  full_count         p95         p05  window_count  anomalies_ge_p95_count        earliest_ge_p95_ts  anomalies_le_p05_count        earliest_le_p05_ts
0    ts-auth-service                trace.from_root.duration_mean          25    0.413442    0.255286            25                       2 2024-01-23 11:24:00+00:00                       2 2024-01-23 11:26:00+00:00
1    ts-auth-service                 trace.from_root.duration_p95          25    1.019867    0.445717            25                       2 2024-01-23 11:32:00+00:00                       2 2024-01-23 11:37:00+00:00
2    ts-auth-service                   trace.from_root.error_rate          25    0.000000    0.000000            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
3    ts-auth-service                    trace.from_root.row_count          25   88.800000   65.200000            25                       2 2024-01-23 11:38:00+00:00                       2 2024-01-23 11:24:00+00:00
4    ts-auth-service     trace.from_ts-auth-service.duration_mean          25    0.004314    0.001978            25                       2 2024-01-23 11:24:00+00:00                       2 2024-01-23 11:32:00+00:00
..               ...                                          ...         ...         ...         ...           ...                     ...                       ...                     ...                       ...
57  ts-route-service    trace.from_ts-travel-service.duration_p95          25    0.004205    0.003692            25                       2 2024-01-23 11:40:00+00:00                       2 2024-01-23 11:24:00+00:00
58  ts-route-service      trace.from_ts-travel-service.error_rate          25    0.000000    0.000000            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
59  ts-route-service       trace.from_ts-travel-service.row_count          25  345.600000  216.000000            25                       2 2024-01-23 11:26:00+00:00                       2 2024-01-23 11:24:00+00:00
60  ts-route-service  trace.from_ts-travel2-service.duration_mean          25    0.003218    0.002943            25                       2 2024-01-23 11:37:00+00:00                       2 2024-01-23 11:28:00+00:00
61  ts-route-service   trace.from_ts-travel2-service.duration_p95          25    0.003967    0.003569            25                       2 2024-01-23 11:38:00+00:00                       2 2024-01-23 11:28:00+00:00

[50 rows x 10 columns], 0                      trace.from_root.duration_mean
1                       trace.from_root.duration_p95
2                         trace.from_root.error_rate
3                          trace.from_root.row_count
4           trace.from_ts-auth-service.duration_mean
                           ...                      
83    trace.from_ts-inside-payment-service.row_count
84         trace.to_ts-station-service.duration_mean
85          trace.to_ts-station-service.duration_p95
86            trace.to_ts-station-service.error_rate
87             trace.to_ts-station-service.row_count
Name: trace_name, Length: 88, dtype: object,              cmdb_id         log_name  full_count     p95     p05  window_count  anomalies_ge_p95_count        earliest_ge_p95_ts  anomalies_le_p05_count        earliest_le_p05_ts
0    ts-auth-service  log.error_count          25     0.0     0.0            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
1    ts-auth-service    log.row_count          25   352.0   266.0            25                       3 2024-01-23 11:34:00+00:00                       2 2024-01-23 11:24:00+00:00
2   ts-order-service  log.error_count          25     0.0     0.0            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
3   ts-order-service    log.row_count          25   469.8   269.8            25                       2 2024-01-23 11:26:00+00:00                       2 2024-01-23 11:24:00+00:00
4   ts-route-service  log.error_count          25     0.0     0.0            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
5   ts-route-service    log.row_count          25  1129.4   808.4            25                       2 2024-01-23 11:42:00+00:00                       2 2024-01-23 11:24:00+00:00
6  ts-travel-service  log.error_count          25     0.0     0.0            25                      25 2024-01-23 11:24:00+00:00                      25 2024-01-23 11:24:00+00:00
7  ts-travel-service    log.row_count          25  2073.4  1227.2            25                       2 2024-01-23 11:26:00+00:00                       2 2024-01-23 11:24:00+00:00, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing metric_df and services if present; otherwise load/define
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

services = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Ensure timestamps and numeric values (idempotent if already done)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 11:21:47", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 11:51:47", tz="UTC")

# 1) Subset to services and compute global thresholds (full series) per (cmdb_id, kpi_name)
metric_subset = metric_df[metric_df['cmdb_id'].isin(services)].copy()

metric_thresholds_full = (
    metric_subset
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(full_count='count',
         p95=lambda x: x.quantile(0.95),
         p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window and merge thresholds for per-row comparison
metric_window = metric_subset[(metric_subset['timestamp'] >= window_start) & (metric_subset['timestamp'] <= window_end)].copy()
metric_window = metric_window.merge(metric_thresholds_full[['cmdb_id','kpi_name','p95','p05']], on=['cmdb_id','kpi_name'], how='left')

# Create boolean flags for breaches
metric_window['is_high'] = metric_window['value'] >= metric_window['p95']
metric_window['is_low']  = metric_window['value'] <= metric_window['p05']

# Helper to extract consecutive runs of a boolean flag per (cmdb_id,kpi_name)
def extract_runs(df, flag_col, polarity_label):
    # df expected sorted by timestamp
    if df.empty:
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','polarity','fault_start_ts','fault_end_ts','fault_length',
            'max_or_min_value','p95','p05'
        ])
    df = df.sort_values('timestamp').reset_index(drop=True)
    # define run ids where flag changes (based on full series order)
    flag_series = df[flag_col].fillna(False).astype(bool)
    run_id = (flag_series != flag_series.shift(1)).cumsum()
    df['run_id'] = run_id
    # keep only runs where flag is True
    true_runs = df[flag_series].groupby('run_id', dropna=False)
    rows = []
    for rid, grp in true_runs:
        length = len(grp)
        if length >= 2:  # keep only runs with length >= 2
            start_ts = grp['timestamp'].min()
            end_ts   = grp['timestamp'].max()
            if polarity_label == 'high':
                max_or_min = grp['value'].max()
            else:
                max_or_min = grp['value'].min()
            # thresholds are same within group; take first
            p95 = grp['p95'].iloc[0]
            p05 = grp['p05'].iloc[0]
            rows.append({
                'cmdb_id': grp['cmdb_id'].iloc[0],
                'kpi_name': grp['kpi_name'].iloc[0],
                'polarity': polarity_label,
                'fault_start_ts': start_ts,
                'fault_end_ts': end_ts,
                'fault_length': int(length),
                'max_or_min_value': float(max_or_min),
                'p95': float(p95) if not pd.isna(p95) else np.nan,
                'p05': float(p05) if not pd.isna(p05) else np.nan
            })
    return pd.DataFrame(rows)

# Apply extraction per group for both polarities and store full results
groups = metric_window.groupby(['cmdb_id','kpi_name'], dropna=False)
high_runs_list = []
low_runs_list = []
for (cmdb_id,kpi_name), grp in groups:
    # Need to pass a sorted copy
    g = grp.sort_values('timestamp')
    high_df = extract_runs(g, 'is_high', 'high')
    low_df  = extract_runs(g, 'is_low', 'low')
    if not high_df.empty:
        high_runs_list.append(high_df)
    if not low_df.empty:
        low_runs_list.append(low_df)

if high_runs_list:
    high_runs_df = pd.concat(high_runs_list, ignore_index=True)
else:
    high_runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','polarity','fault_start_ts','fault_end_ts','fault_length',
        'max_or_min_value','p95','p05'
    ])

if low_runs_list:
    low_runs_df = pd.concat(low_runs_list, ignore_index=True)
else:
    low_runs_df = pd.DataFrame(columns=high_runs_df.columns)

# Compute breach ratios:
# For 'high': (max_value_in_run - p95) / p95
# For 'low' : (p95 - min_value_in_run) / p95  (use same p95 reference as requested)
def compute_breach_ratios(df):
    if df.empty:
        df['breach_ratio'] = pd.Series(dtype=float)
        return df
    # Avoid division by zero: if p95 == 0 -> set breach_ratio = NaN
    df = df.copy()
    df['p95_nonzero'] = df['p95'].replace({0: np.nan})
    # For high polarity, use max_or_min_value as max; for low, it's min
    df['breach_ratio'] = np.where(
        df['polarity'] == 'high',
        (df['max_or_min_value'] - df['p95_nonzero']) / df['p95_nonzero'],
        (df['p95_nonzero'] - df['max_or_min_value']) / df['p95_nonzero']
    )
    # If p95 was zero, breach_ratio will be NaN; keep as-is
    df = df.drop(columns=['p95_nonzero'])
    return df

high_runs_df = compute_breach_ratios(high_runs_df)
low_runs_df  = compute_breach_ratios(low_runs_df)

# Combine both polarities
faults_full = pd.concat([high_runs_df, low_runs_df], ignore_index=True, sort=False)

# Create a sort key: for high use breach_ratio, for low also use breach_ratio (we computed for low accordingly)
# For ranking, NaNs sort last; replace NaN with -inf to push to bottom when sorting descending
faults_full['sort_key'] = faults_full['breach_ratio'].fillna(-np.inf)

# Sort by sort_key descending and then by fault_length descending, keep top 20
faults_top20 = faults_full.sort_values(['sort_key','fault_length'], ascending=[False, False]).head(20)

# For display, rename max_or_min_value column to 'max_or_min_value' already set; ensure timestamp columns are timezone-aware
faults_top20['fault_start_ts'] = pd.to_datetime(faults_top20['fault_start_ts'], utc=True)
faults_top20['fault_end_ts']   = pd.to_datetime(faults_top20['fault_end_ts'], utc=True)

# Short aggregate: distinct cmdb_id values that have at least one fault and count of faults per cmdb_id
if not faults_full.empty:
    fault_counts_per_cmdb = faults_full.groupby('cmdb_id', dropna=False).size().reset_index(name='fault_count').sort_values('fault_count', ascending=False)
    fault_cmdb_list = pd.Series(fault_counts_per_cmdb['cmdb_id'].unique(), name='cmdb_id')
else:
    fault_counts_per_cmdb = pd.DataFrame(columns=['cmdb_id','fault_count'])
    fault_cmdb_list = pd.Series([], name='cmdb_id', dtype=object)

# Final compact table columns as requested
faults_top20_table = faults_top20[[
    'cmdb_id','kpi_name','polarity','fault_start_ts','fault_end_ts','fault_length',
    'max_or_min_value','p95','p05','breach_ratio'
]].reset_index(drop=True)

# Store full intermediate results in kernel variables for later use
metric_thresholds_full = metric_thresholds_full
metric_window = metric_window
high_runs_df = high_runs_df
low_runs_df = low_runs_df
faults_full = faults_full

# Output compact results (DataFrame and aggregates)
faults_top20_table, fault_cmdb_list, fault_counts_per_cmdb.head(20)

```
Out[3]:
```
```python
# Build a concise plain-English summary from existing kernel variables (faults_top20_table, fault_counts_per_cmdb)
top_faults = faults_top20_table.head(5)
counts = fault_counts_per_cmdb

lines = []
lines.append("Summary of metric fault runs (services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service)")
lines.append("Incident window: 2024-01-23 11:21:47 to 2024-01-23 11:51:47 UTC")
lines.append("")
lines.append("Top consecutive fault runs (at least 2 consecutive points):")
for i, row in top_faults.iterrows():
    br = row['breach_ratio']
    br_s = "NaN" if pd.isna(br) else f"{br:.3f}"
    lines.append(
        f"- {row['cmdb_id']} | {row['kpi_name']} | polarity={row['polarity']} | "
        f"start={row['fault_start_ts'].isoformat()} | end={row['fault_end_ts'].isoformat()} | "
        f"length={int(row['fault_length'])} | value={row['max_or_min_value']:.6g} | p95={row['p95']:.6g} | breach_ratio={br_s}"
    )

lines.append("")
lines.append("Interpretation of top findings:")
lines.append("- The largest breaches (by relative breach_ratio) are 'low' dips on ts-train-service (cpu, latency-90, mem) around 11:28–11:29 and 11:24–11:25 UTC.")
lines.append("- There are both 'low' polarity runs (values falling at-or-below P05) and some 'high' polarity runs (values exceeding P95).")
lines.append("- Memory values for several services show minor relative deviations; absolute memory p95 values are large but relative breaches are small for many groups.")
lines.append("- Some 'error' KPIs have p95 = 0 which yields NaN breach_ratio; those reflect constant-zero error rates rather than actionable spikes.")

lines.append("")
lines.append("Fault counts per service (number of detected runs):")
for _, r in counts.iterrows():
    lines.append(f"- {r['cmdb_id']}: {int(r['fault_count'])} faults")

lines.append("")
lines.append("Recommendation:")
lines.append("- Focus initial investigation on ts-train-service (multiple high-ranked dips) and ts-route-service (multiple faults). Check application and infra logs at the reported start timestamps (UTC) to correlate these runs.")
lines.append("- Ignore constant-zero error series as failure signals; concentrate on CPU/memory/latency dips and socket spikes reported in the runs.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name polarity            fault_start_ts              fault_end_ts  fault_length  max_or_min_value           p95           p05  breach_ratio
0    ts-train-service         cpu      low 2024-01-23 11:28:00+00:00 2024-01-23 11:29:00+00:00             2      2.506629e+00  3.064215e+01  2.519470e+00      0.918197
1    ts-train-service  latency-90      low 2024-01-23 11:28:00+00:00 2024-01-23 11:29:00+00:00             2      8.854752e-03  6.818233e-02  8.869189e-03      0.870131
2    ts-train-service         mem      low 2024-01-23 11:24:00+00:00 2024-01-23 11:25:00+00:00             2      2.424971e+08  1.031396e+09  2.440089e+08      0.764885
3    ts-route-service  latency-90      low 2024-01-23 11:28:00+00:00 2024-01-23 11:29:00+00:00             2      4.895987e-03  7.261232e-03  4.914381e-03      0.325736
4    ts-route-service         cpu      low 2024-01-23 11:28:00+00:00 2024-01-23 11:29:00+00:00             2      3.371002e+00  4.591703e+00  3.619001e+00      0.265849
5   ts-travel-service  latency-50      low 2024-01-23 11:27:00+00:00 2024-01-23 11:28:00+00:00             2      1.316158e-02  1.746523e-02  1.379154e-02      0.246413
6    ts-route-service      socket      low 2024-01-23 11:33:00+00:00 2024-01-23 11:34:00+00:00             2      1.218333e+01  1.393333e+01  1.262000e+01      0.125598
7    ts-route-service  latency-50      low 2024-01-23 11:28:00+00:00 2024-01-23 11:29:00+00:00             2      3.164437e-03  3.416091e-03  3.174656e-03      0.073667
8   ts-travel-service  latency-50     high 2024-01-23 11:38:00+00:00 2024-01-23 11:39:00+00:00             2      1.798637e-02  1.746523e-02  1.379154e-02      0.029839
9    ts-order-service         mem      low 2024-01-23 11:24:00+00:00 2024-01-23 11:25:00+00:00             2      2.601596e+08  2.637848e+08  2.602987e+08      0.013743
10    ts-auth-service         mem      low 2024-01-23 11:24:00+00:00 2024-01-23 11:25:00+00:00             2      2.368121e+08  2.399867e+08  2.369201e+08      0.013228
11   ts-train-service      socket     high 2024-01-23 11:45:00+00:00 2024-01-23 11:46:00+00:00             2      2.130000e+01  2.105333e+01  1.667333e+01      0.011716
12  ts-travel-service         mem      low 2024-01-23 11:24:00+00:00 2024-01-23 11:25:00+00:00             2      2.707866e+08  2.730814e+08  2.710320e+08      0.008403
13   ts-order-service  latency-50     high 2024-01-23 11:29:00+00:00 2024-01-23 11:30:00+00:00             2      7.671449e-03  7.615756e-03  6.233425e-03      0.007313
14  ts-travel-service         mem     high 2024-01-23 11:42:00+00:00 2024-01-23 11:43:00+00:00             2      2.743773e+08  2.730814e+08  2.710320e+08      0.004746
15   ts-train-service         cpu     high 2024-01-23 11:45:00+00:00 2024-01-23 11:46:00+00:00             2      3.070624e+01  3.064215e+01  2.519470e+00      0.002091
16   ts-route-service         mem      low 2024-01-23 11:24:00+00:00 2024-01-23 11:25:00+00:00             2      2.555781e+08  2.558868e+08  2.555854e+08      0.001206
17    ts-auth-service         mem     high 2024-01-23 11:47:00+00:00 2024-01-23 11:48:00+00:00             2      2.400485e+08  2.399867e+08  2.369201e+08      0.000258
18   ts-order-service       error     high 2024-01-23 11:24:00+00:00 2024-01-23 11:48:00+00:00            25      0.000000e+00  0.000000e+00  0.000000e+00           NaN
19   ts-order-service       error      low 2024-01-23 11:24:00+00:00 2024-01-23 11:48:00+00:00            25      0.000000e+00  0.000000e+00  0.000000e+00           NaN, 0     ts-route-service
1     ts-train-service
2     ts-order-service
3    ts-travel-service
4      ts-auth-service
Name: cmdb_id, dtype: object,              cmdb_id  fault_count
2   ts-route-service            5
3   ts-train-service            5
1   ts-order-service            4
4  ts-travel-service            4
0    ts-auth-service            2)```
```

In [ ]:
# In[4]:

import re
import pandas as pd

# Reuse existing trace_df if present; otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df.get('value', pd.Series()), errors='coerce')

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-23 11:21:47", tz="UTC")
window_end   = pd.Timestamp("2024-01-23 11:51:47", tz="UTC")

# Candidate services set
candidates = {"ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"}

# Filter traces to the incident window
trace_window = trace_df[(trace_df['timestamp'] >= window_start) & (trace_df['timestamp'] <= window_end)].copy()

# Function to extract ts-...-service tokens from a trace_name
service_pattern = re.compile(r"(ts-[a-z0-9\-]+-service)")

def extract_services(name):
    if pd.isna(name):
        return []
    found = service_pattern.findall(str(name))
    # normalize and deduplicate while preserving sorted order
    uniq = list(dict.fromkeys(found))
    return uniq

# Aggregate per trace_name:
# - involved_services: unique services found inside trace_name
# - emitters: unique cmdb_id values seen in window for that trace_name
# - window_count: number of rows in window for that trace_name
# - earliest_ts: earliest timestamp in window for that trace_name
agg = (
    trace_window
    .groupby('trace_name', dropna=False)
    .agg(
        window_count=('trace_name','count'),
        emitters_list=('cmdb_id', lambda s: sorted(pd.Series(s).dropna().unique().tolist())),
        earliest_ts=('timestamp','min')
    )
    .reset_index()
)

# Compute involved_services by extracting from trace_name string
agg['involved_services'] = agg['trace_name'].apply(extract_services)

# Keep only rows where involved_services intersects candidate set
def intersect_candidates(lst):
    return sorted(list(set(lst) & candidates))

agg['involved_candidates'] = agg['involved_services'].apply(intersect_candidates)
agg_filtered = agg[agg['involved_candidates'].map(len) > 0].copy()

# Rearrange columns and sort by window_count desc, limit to top 50
trace_summary = agg_filtered[['trace_name','involved_services','emitters_list','window_count','earliest_ts']].sort_values('window_count', ascending=False).head(50)
trace_summary = trace_summary.rename(columns={'emitters_list':'emitters','earliest_ts':'earliest_ts'})

# Aggregate: top 10 trace_names that include >=2 of the candidate services
multi_candidates = (
    agg_filtered[agg_filtered['involved_candidates'].map(len) >= 2]
    .assign(involved_candidates=lambda df: df['involved_candidates'].apply(lambda x: sorted(x)))
    .loc[:, ['trace_name','involved_candidates','window_count']]
    .sort_values('window_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Final compact outputs
trace_summary, multi_candidates

```
Out[4]:
```
```python
# Compose a concise plain-English summary string using the analysis results already in kernel variables:
summary_lines = []
summary_lines.append("Trace-window summary (2024-01-23 11:21:47 to 2024-01-23 11:51:47 UTC):")
summary_lines.append("")
# number of trace_name rows returned
try:
    n_rows = len(trace_summary)
except NameError:
    n_rows = 0
summary_lines.append(f"- Trace names involving candidate services found in window: {n_rows} (top 50 shown).")
summary_lines.append("")
# Top examples
if n_rows > 0:
    # top trace by window_count (first row)
    top_row = trace_summary.iloc[0]
    top_name = top_row['trace_name']
    top_involved = top_row['involved_services']
    top_emitters = top_row['emitters']
    top_count = int(top_row['window_count'])
    top_earliest = pd.to_datetime(top_row['earliest_ts'], utc=True)
    summary_lines.append(f"- Top trace: {top_name} -> involved_services = {top_involved}, emitters sample = {top_emitters[:6]} (showing up to 6), window_count = {top_count}, earliest_ts = {top_earliest.isoformat()}.")
    # second example if exists
    if n_rows > 1:
        row2 = trace_summary.iloc[1]
        summary_lines.append(f"- 2nd: {row2['trace_name']} -> involved_services = {row2['involved_services']}, window_count = {int(row2['window_count'])}, earliest_ts = {pd.to_datetime(row2['earliest_ts'], utc=True).isoformat()}.")
    # third example if exists
    if n_rows > 2:
        row3 = trace_summary.iloc[2]
        summary_lines.append(f"- 3rd: {row3['trace_name']} -> involved_services = {row3['involved_services']}, window_count = {int(row3['window_count'])}.")
else:
    summary_lines.append("- No matching trace names found in the window for the candidate services.")

summary_lines.append("")
# multi-candidate traces
try:
    multi_count = len(multi_candidates)
except NameError:
    multi_count = 0
if multi_count == 0:
    summary_lines.append("- No trace_name in the window references two or more of the five candidate services simultaneously.")
else:
    summary_lines.append(f"- {multi_count} trace_names reference >=2 candidate services (top 10 shown).")

summary_lines.append("")
summary_lines.append("Interpretation and recommendation:")
summary_lines.append("- Many trace entries reference a single candidate service (e.g., trace.to_ts-order-service.* with high window_count), indicating traffic/observability focused on that target service.")
summary_lines.append("- The top traces by window_count are: traces targeting ts-order-service (multiple metrics: duration_mean, duration_p95, error_rate, row_count) and traces from ts-travel-service. Investigate those emitters and earliest timestamps (UTC) for correlation with any incidents.")
summary_lines.append("- Because no trace_name included >=2 candidate services, there is no immediate multi-service trace pattern tying multiple candidate services together in this window.")
summary_lines.append("")
summary_lines.append("If you want, I can list the exact top 20 trace_name rows (trace_name, involved_services, emitters, window_count, earliest_ts) for direct inspection.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                     trace_name    involved_services                                           emitters  window_count               earliest_ts
164     trace.to_ts-order-service.duration_mean   [ts-order-service]  [root, ts-inside-payment-service, ts-order-ser...           173 2024-01-23 11:24:00+00:00
165      trace.to_ts-order-service.duration_p95   [ts-order-service]  [root, ts-inside-payment-service, ts-order-ser...           173 2024-01-23 11:24:00+00:00
166        trace.to_ts-order-service.error_rate   [ts-order-service]  [root, ts-inside-payment-service, ts-order-ser...           173 2024-01-23 11:24:00+00:00
167         trace.to_ts-order-service.row_count   [ts-order-service]  [root, ts-inside-payment-service, ts-order-ser...           173 2024-01-23 11:24:00+00:00
100  trace.from_ts-travel-service.duration_mean  [ts-travel-service]  [ts-order-service, ts-route-service, ts-seat-s...           150 2024-01-23 11:24:00+00:00
..                                          ...                  ...                                                ...           ...                       ...
79        trace.from_ts-route-service.row_count   [ts-route-service]                                 [ts-route-service]            25 2024-01-23 11:24:00+00:00
96    trace.from_ts-train-service.duration_mean   [ts-train-service]                                 [ts-train-service]            25 2024-01-23 11:24:00+00:00
97     trace.from_ts-train-service.duration_p95   [ts-train-service]                                 [ts-train-service]            25 2024-01-23 11:24:00+00:00
98       trace.from_ts-train-service.error_rate   [ts-train-service]                                 [ts-train-service]            25 2024-01-23 11:24:00+00:00
99        trace.from_ts-train-service.row_count   [ts-train-service]                                 [ts-train-service]            25 2024-01-23 11:24:00+00:00

[40 rows x 5 columns], Empty DataFrame
Columns: [trace_name, involved_candidates, window_count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing trace_summary and candidates from kernel
# Defensive checks: ensure trace_summary exists
try:
    trace_summary
except NameError:
    raise RuntimeError("trace_summary not found in kernel. Run the trace summary step first.")

# Helper to coerce possible NaNs into empty lists
def to_list_safe(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if pd.isna(x):
        return []
    # if it's a string representation of list, attempt eval (but avoid eval); return single-item list
    return [x]

# Build emitters_intersection and involved_candidates columns safely
trace_enriched = trace_summary.copy()
trace_enriched['emitters'] = trace_enriched['emitters'].apply(to_list_safe)
trace_enriched['involved_services'] = trace_enriched['involved_services'].apply(to_list_safe)

trace_enriched['emitters_intersection'] = trace_enriched['emitters'].apply(lambda lst: sorted(list(set(lst) & candidates)))
trace_enriched['involved_candidates'] = trace_enriched['involved_services'].apply(lambda lst: sorted(list(set(lst) & candidates)))

# Condition: at least one emitter in candidates AND involved_candidates contains at least one candidate different from the emitter(s)
def qualifies_row(row):
    e = set(to_list_safe(row.get('emitters_intersection', [])))
    i = set(to_list_safe(row.get('involved_candidates', [])))
    if not e or not i:
        return False
    return len(i - e) > 0  # at least one involved candidate different from emitter(s)

trace_enriched['qualifies'] = trace_enriched.apply(qualifies_row, axis=1)

# Select matches and compact output
matches = trace_enriched[trace_enriched['qualifies']].copy()
matches_out = matches[['trace_name','emitters_intersection','involved_candidates','window_count','earliest_ts']].sort_values('window_count', ascending=False).head(50).reset_index(drop=True)

# Aggregate: distinct emitter candidate services and count of trace_name rows where they appear as emitters
if not matches_out.empty:
    expl = matches_out[['trace_name','emitters_intersection']].explode('emitters_intersection').dropna(subset=['emitters_intersection'])
    if not expl.empty:
        emit_counts = (
            expl.groupby('emitters_intersection', dropna=False)
            .agg(trace_name_count=('trace_name','nunique'))
            .reset_index()
            .rename(columns={'emitters_intersection':'emitter_candidate'})
            .sort_values('trace_name_count', ascending=False)
            .reset_index(drop=True)
        )
    else:
        emit_counts = pd.DataFrame(columns=['emitter_candidate','trace_name_count'])
else:
    matches_out = pd.DataFrame(columns=['trace_name','emitters_intersection','involved_candidates','window_count','earliest_ts'])
    emit_counts = pd.DataFrame(columns=['emitter_candidate','trace_name_count'])

# If empty, provide explicit empty indicator by returning the empty DataFrame and empty emit_counts
matches_out, emit_counts

```
Out[5]:
```
Summary (plain English):

- I looked for traces in the 2024-01-23 11:21:47–11:51:47 UTC window where:
  1) at least one emitter (cmdb_id) is one of the five candidate services (ts-auth, ts-order, ts-route, ts-train, ts-travel), and
  2) the trace_name's parsed involved_services also includes at least one of the five candidate services that is different from that emitter.

- Result: No matches were found. The query returned an empty table (no trace_name rows meet both conditions) and the emitter-aggregate is also empty.

- Interpretation / next steps:
  - Within the window, candidate services do emit traces and many trace_names reference candidate services (e.g., traces targeting ts-order-service, traces from ts-travel-service), but none of those trace_names show a pattern where a candidate service emitted a trace that references a different candidate service.
  - If you want to broaden the search, consider:
    - Allowing same-service traces (emitters == involved service),
    - Looking at a wider time window, or
    - Inspecting traces where emitters are non-candidate services but involved_services include multiple candidate services (cross-service calls originating outside the candidate set).

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [trace_name, emitters_intersection, involved_candidates, window_count, earliest_ts]
Index: [], Empty DataFrame
Columns: [emitter_candidate, trace_name_count]
Index: [])```
```